In [ ]:
# educational pathway: 
# single-head -> 



# why multi-head is better than single-head ? 

# https://gemini.google.com/app/8720e969a2fcbae5
# The Averaging Bottleneck (Single Head)
# Representation Subspaces (Multi-Head)
# The Computational Equivalence

# too many heads:
#   Dot-product discriminability collapses: In a d_head-dimensional random projection, the variance of the dot product between two vectors scales with d_head, When d_head gets very small, there just isn't enough room for queries/keys to be pushed far apart in angle — random or near-random keys start looking similar to a query, so the softmax degenerates toward uniform, You get noise-induced averaging, which is the same symptom you were trying to escape (uniform attention), just caused by insufficient dimensionality instead of a forced single softmax.
#   Johnson–Lindenstrauss-style capacity limit. To keep pairwise angular relationships between N distinguishable concepts roughly preserved under a linear projection, you need roughly d_head = O(log N / ε²) dimensions. Below that, key vectors for genuinely different tokens start colliding in the projected space — the head physically cannot keep enough patterns separable   


# pay-off curve = inverted-U
# 
# performance
    # ^
    # |         ____________
    # |       /              \___
    # |      /                    \___
    # |     /                          \
    # |    /
    # |   /
    # +--+---------------------------------> h (num heads, d_head = d_model/h)
    #    1    few heads    "sweet spot"    many tiny heads
    #    (averaging       (diverse, still  (noisy dot products,
    #     bottleneck)      well-powered)    redundant/collapsing heads)
#  


# Multi-head splitting trades a combinatorial bottleneck (can't represent multiple relations at once) for a dimensionality/statistical 
# bottleneck (each relation is represented too weakly to be reliably distinguished). The optimum sits where you've captured most of the 
# useful "different relation types need different heads" benefit before dot-product signal-to-noise per head starts degrading — empirically, 
# that's around d_head ≈ 64–128 for typical model/vocab scales, largely independent of total model size.


#---
# implement YaRN for positinal encodings, QK norm

# MHA: num_query_heads == num_kv_heads
# GQA: 1 < num_kv_heads < num_query_heads and num_query_heads % num_kv_heads == 0
# MQA: num_kv_heads == 1


# TODO:
# tunable knobs of q/kv heads(single, grouped,multi), YaRN, QK norm, KV caching


In [ ]:
import os
import sys
import math
import time
import torch
import random
import numpy as np
import torch.nn as nn
from pathlib import Path
import torch.nn.functional as F
import matplotlib.pyplot as plt
from einops import rearrange, reduce, repeat, einsum, pack


import training_engine as training_engine 
from inference_engine import advanced_inference, KV_cache
from utils import RMSNorm, init_custom_weight, inspect_tensor


MODEL_CONFIGS_lvl2 = {

    "embed_dim"   : 2**5,   # 32

    "max_context_window" : 2**6,   # 64
    
    "vocab_size"  : 2**11,  # 2048
    
    "num_q_heads" : 2**2, # 4 | 1, 1, 2, 4
    "num_kv_heads": 2**2, # 4 | 1, 4, 4, 4

    "num_layers": 4,
   
    "max_batch_size": 2**4,   # 16
   
    "learning_rate" : 1e-3,

    "save_path" :    "../models/level3_attention_models.pt",
    "loss_history" : "../models/level3_attention_models_loss_history.txt",
}


# Device configuration
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)


cuda


In [ ]:
class CausalAttention_SubBlock(nn.Module):
    def __init__(self, config=MODEL_CONFIGS_lvl2):
        super().__init__()

        # assert config["num_q_heads"] % config["num_kv_heads"] == 0
        # head_dim = embed_dim // num_q_heads
        # W_q , W_k , W_v , W_o 
        # kv buffers
        # use -> F.scaled_dot_product_attention(..., is_causal=True)

        self.embed_dim = config["embed_dim"]
        self.num_attn_heads = config["num_attn_heads"]
        self.num_layers = config["num_layers"] 

        assert self.embed_dim % self.num_attn_heads == 0, \
            "embed_dim must be divisible by num_attn_heads"

        self.head_dim = self.embed_dim // self.num_attn_heads


        self.W_q = nn.Parameter(torch.empty(self.embed_dim, self.num_attn_heads * self.head_dim))
        self.W_k = nn.Parameter(torch.empty(self.embed_dim, self.num_attn_heads * self.head_dim))
        self.W_v = nn.Parameter(torch.empty(self.embed_dim, self.num_attn_heads * self.head_dim))

        self.W_o   = nn.Parameter(torch.empty(self.num_attn_heads * self.head_dim, self.embed_dim))

        self.init_params()




        # init KV cache as pre-allocated fixed-size tensor buffer [batch, n_kv_heads, max_seq_len, head_dim] 

    def init_params(self):
        # Standard projections
        init_custom_weight(self.W_q, is_residual_output=False)
        init_custom_weight(self.W_k, is_residual_output=False)
        init_custom_weight(self.W_v, is_residual_output=False)

        # Residual output projection (scaled by depth)
        init_custom_weight(self.W_o, is_residual_output=True, num_layers=self.num_layers)


    def apply_RoPE(self, x):
        ...

    def forward(self, 
                x, 
                cache: Optional[KVCache] = None, 
                layer_idx:int = 0):
        batch_size, seq_len, embed_dim = x.shape

        q = einsum(
            x, self.query_w,
            "batch_size seq_len embed_dim, "
            "embed_dim num_attn_heads_times_head_dim " 
            " -> "
            "batch_size seq_len num_attn_heads_times_head_dim",
        )

        k = einsum(
            x, self.key_w,
            "batch_size seq_len embed_dim, "
            "embed_dim num_attn_heads_times_head_dim " 
            " -> "
            "batch_size seq_len num_attn_heads_times_head_dim",
        )

        v = einsum(
            x, self.value_w,
            "batch_size seq_len embed_dim, "
            "embed_dim num_attn_heads_times_head_dim " 
            " -> "
            "batch_size seq_len num_attn_heads_times_head_dim",
        )


        q = rearrange(
            q,
            "batch_size seq_len (num_attn_heads head_dim) " 
            " -> "
            "batch_size num_attn_heads seq_len head_dim",
            num_attn_heads=self.num_attn_heads,
        )
        k = rearrange(
            k,
            "batch_size seq_len (num_attn_heads head_dim) " 
            " -> "
            "batch_size num_attn_heads seq_len head_dim",
            num_attn_heads=self.num_attn_heads,
        )
        v = rearrange(
            v,
            "batch_size seq_len (num_attn_heads head_dim) " 
            " -> "
            "batch_size num_attn_heads seq_len head_dim",
            num_attn_heads=self.num_attn_heads,
        )

        # project through R(pos) matrix-rotate for RoPE
        # cos/sin are ALREADY sliced to the absolute positions [start_pos : start_pos+t]
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)

        
        # store in KV cache
        if cache is not None:
            k, v = cache.update(layer_idx, k, v)   # cache handles the cat / storage
        # now do attention with full k, v

        # attention scores Q_T K / d**-0.5
        atten_scores = einsum(
            q, k,
            "batch_size num_attn_heads query_len head_dim, "
            "batch_size num_attn_heads key_len head_dim " 
            " -> "
            "batch_size num_attn_heads query_len key_len",
        ) 

        variance_scaled_atten_scores = atten_scores / math.sqrt(self.head_dim)
        causaly_masked_atten_scores  = ... 

        atten_weights = F.softmax(causaly_masked_atten_scores, dim=-1)

        atten_weighted_value_message_passing = einsum(
            atten_weights, v,
            "batch_size num_attn_heads query_len key_len, "
            "batch_size num_attn_heads seq_len head_dim " 
            " -> "
            "batch_size num_attn_heads ",
        ) 

        output = einsum(
            atten_weighted_value_message_passing, self.W_o,
            ""
            ""
            " -> "
            "",
        )


        return output



In [ ]:
class Level2Transformer_Block(nn.Module):
    def __init__(self, config=MODEL_CONFIGS_lvl2):
        super().__init__()
        # init rms_1 and attn
        # init rms_2 and mlp(not needed in lvl 3)

    def forward(self, x, cache: Optional[KVCache] = None, layer_idx: int = 0):
        # store residual
        atten_residual = x

        # appy rms_1 then atten
        # add residual to result 
        
        return x


In [ ]:
class Level2CausalAttention_Model(nn.Module):

    def __init__(self, config=MODEL_CONFIGS_lvl2):
        super().__init__()

        # init embed, final_rms and unemded 
        # init num_layer of Blocks


        mask = torch.triu(
            torch.ones(config["max_seq_len"], config["max_seq_len"], dtype=torch.bool),
            diagonal=1,
        )
        self.register_buffer("mask", mask, persistent=False)


    def init_weights(self):
        init_custom_weight(self.embed, is_residual_output=False)
        init_custom_weight(self.un_embed, is_residual_output=False)


    def forward(self, idx, cache: Optional[KVCache] = None,):

        batch_size, seq_len = idx.shape

        for i, layer in enumerate(self.layers):   # your blocks
            x = layer(x, cache=cache, layer_idx=i)
        # 


        return logits

In [ ]:


model = Level3SingleHeadCausalAttention_Model()
configs = MODEL_CONFIGS_lvl3

training_engine.train_and_save_model(model, configs, device, 100)
training_engine.plot_loss_history(configs)


if torch.cuda.is_available():
    torch.cuda.synchronize()
start = time.time()

    # token_ids = generate_text_simple_cached(
    #     model=model,
    #     idx=encoded_tensor,
    #     max_new_tokens=args.max_new_tokens,
    # )

if torch.cuda.is_available():
    torch.cuda.synchronize()
total_time = time.time() - start



if torch.cuda.is_available():
    max_mem_bytes = torch.cuda.max_memory_allocated()
    max_mem_gb = max_mem_bytes / (1024 ** 3)
    print(f"Max memory allocated: {max_mem_gb:.2f} GB")


In [ ]:
eos_id = 83

# use "continuous batching" to pack same-length prompts together 
prompt_strs = [
    "write a story about",
    "write like a little boy that was angry",
    "write as a little boy that was happy,",
] # 3 prompts each w/ len of 8.

result = inference_engine.advanced_inference(model, configs, num_get_steps=5, prompt_strs=prompt_strs, 
                                             device=device, eos_token_id=eos_id) # [batch_size, prompt_len + num_get_steps]

for prmpt in result:
    print("---"*80)
    print(prmpt)


------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
write a story about
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
write like a little boy that was angry
------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
write as a little boy that was happy,
